In [ ]:
from notebook_setup import setup
from vizman import viz
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pandas as pd

from matplotlib import font_manager
for font in font_manager.findSystemFonts("figures/Atkinson_Typeface/"):
    font_manager.fontManager.addfont(font)

viz.set_visual_style()
default_sizes = viz.load_data_from_json("sizes.json")
default_colors = viz.load_data_from_json("colors.json")
default_cmaps = viz.give_colormaps()

from kaysons_visual_config import * # Kayson's file. 

from src.visualization.find_and_plot_closest_connectome import find_closest_connectome, plot_connectome_blockwise, compare_orderings

env = setup()

import numpy as np
from scipy.spatial.distance import squareform
from scipy.optimize import minimize
import networkx as nx
from netgraph import Graph
import matplotlib.pyplot as plt
from pypalettes import load_cmap

from pypalettes import load_cmap
cmap = load_cmap("Aurora")

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from netgraph import Graph
from networkx.algorithms import community



In [ ]:
subject_id = 206
experiment_name = "76_90000_samples_animal_206" 
base_path = Path("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/output/gnm/suarez_MaMI_dataset") / experiment_name

data_dir = base_path / "generated_networks"
save_dir = base_path / "generated_networks_visualizations"
save_dir.mkdir(parents=True, exist_ok=True)

margin_x = 0.5
margin_y = 0.05
interesting_eta_and_gamma_combinations = [
    [-8+margin_x, 1-margin_y],  
    [3-margin_x, 1-margin_y], 
    [-0.8, 0.18],
    [3-margin_x, -0.1+margin_y], 
    [-8+margin_x, -0.1+margin_y],
    [-5, 0.5],

]

# Choose clustering method: 'spectral', 'hierarchical', 'modularity', 'degree', or None
clustering_method = 'spectral'

# Store connectomes and parameters
connectomes = []
params = []

for parameter_combination in interesting_eta_and_gamma_combinations:
    target_eta = parameter_combination[0]
    target_gamma = parameter_combination[1]
    
    found_eta, found_gamma, connectome = find_closest_connectome(target_eta, target_gamma, data_dir)
    
    connectomes.append(connectome)
    params.append((found_eta, found_gamma))
    
    parameter_combination.append(found_eta)
    parameter_combination.append(found_gamma)


In [ ]:
distance_matrices = np.load("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/data/preprocessed/suarez_MaMI_dataset/02_distance_matrices/distance_matrix_50.npy")
d = distance_matrices[subject_id]  

# Normalize distance matrix
d = (d - np.min(d)) / (np.max(d) - np.min(d))

# ============ Force-directed layout with distance weights ============

print("Creating distance-weighted graph for layout...")

# Create a complete weighted graph where edge weights represent desired distances
n_nodes = len(d)
G_distance = nx.Graph()
G_distance.add_nodes_from(range(n_nodes))
print(G_distance)
# Add edges with weights inversely proportional to distances
# (shorter distance = stronger attraction = higher weight)
epsilon = 1e-10
for i in range(n_nodes):
    for j in range(i+1, n_nodes):
        # Convert distance to weight: closer nodes should attract more
        # Using inverse distance as weight
        weight = 1.0 / (d[i, j] + epsilon)
        G_distance.add_edge(i, j, weight=weight)

# # Use spring layout with distance-derived weights
# print("Computing force-directed layout (this may take a moment)...")
# node_layout = nx.kamada_kawai_layout(  # spring_layout(
#     G, # onnectome,  # Use the actual connectome for layout
#     dist = G_distance, 
#     # weight='weight',  # Use the distance-based weights
#     # iterations=1,   # Increase for better convergence
#     # k=None,           # Optimal distance between nodes (auto-calculated)
#     # seed=42           # For reproducibility
# )

# # visualize node layout 
# for idx, (x, y) in enumerate(node_layout.values()):
#     plt.scatter(x, y)

In [ ]:
node_layout = nx.kamada_kawai_layout(  # spring_layout(
    G_distance, # onnectome,  # Use the actual connectome for layout
    # dist = G_distance, 
    # weight='weight',  # Use the distance-based weights
    # iterations=1,   # Increase for better convergence
    # k=None,           # Optimal distance between nodes (auto-calculated)
    # seed=42           # For reproducibility
)


In [ ]:
# import numpy as np

# # Load distance matrix
# subject_id = 206
# distance_matrices = np.load("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/data/preprocessed/suarez_MaMI_dataset/02_distance_matrices/distance_matrix_50.npy")
# d = distance_matrices[subject_id]  

# # Normalize distance matrix
# d = (d - np.min(d)) / (np.max(d) - np.min(d))

# # ============================================
# # Create distance-weighted graph for layout
# # ============================================
# print("Creating distance-weighted graph for layout...")

# n_nodes = len(d)
# G_distance = nx.Graph()
# G_distance.add_nodes_from(range(n_nodes))

# # Add edges with weights inversely proportional to distances
# epsilon = 1e-10
# for i in range(n_nodes):
#     for j in range(i+1, n_nodes):
#         # Convert distance to weight: closer nodes should attract more
#         weight = 1.0 / (d[i, j] + epsilon)
#         G_distance.add_edge(i, j, weight=weight)

# # Compute layout based on distance matrix
# print("Computing force-directed layout based on distance matrix...")
# node_layout = nx.kamada_kawai_layout(G_distance)
# print("Layout computed successfully!")

# # Convert to numpy arrays for compatibility
# for node in node_layout:
#     node_layout[node] = np.array(node_layout[node])

# # ============================================
# # Create CONNECTOME graphs (for edges only)
# # ============================================
# connectome_graphs = []
# for conn in connectomes:
#     G_conn = nx.from_numpy_array(conn)
#     connectome_graphs.append(G_conn)

# # ============================================
# # Extract communities and create node colors
# # ============================================
# from netneurotools.modularity import consensus_modularity as netneurotools_modularity 

# communities_arr = [] 
# node_colors_arr = []
# node_communities_arr = []

# # Load colormap
# cmap = load_cmap("Antique")

# for conn in connectomes:   
#     communities, _, _ = netneurotools_modularity(conn)
#     communities_arr.append(communities)
    
#     # Create node_to_community dictionary
#     node_to_community = {node_id: int(community_id) for node_id, community_id in enumerate(communities)}
    
#     # Get number of unique communities and their sizes
#     unique_communities = np.unique(communities)
#     num_communities = len(unique_communities)
#     partition_sizes = [np.sum(communities == comm) for comm in unique_communities]
    
#     print(f"  Found {num_communities} communities")
#     print(f"  Community sizes: {partition_sizes}")
    
#     # Create color mapping
#     node_color = {node_id: cmap((community_id - 1) / max(num_communities - 1, 1)) 
#                   for node_id, community_id in node_to_community.items()}
    
#     node_colors_arr.append(node_color)
#     node_communities_arr.append(node_to_community)

# # ============================================
# # Create edge properties for EACH connectome
# # ============================================
# labels = ['A', 'B', 'C', 'D', 'E', 'F']
# edge_color_dicts = []
# edge_width_dicts = []
# edge_alpha_dicts = []

# # Define colors for different connection types
# within_left_color = "black"  # Black edges within left hemisphere
# within_right_color = "black"  # Black edges within right hemisphere
# between_hemispheres_color = '#9B59B6'  # Purple for cross-hemisphere connections

# for conn_idx, (conn, G_conn) in enumerate(zip(connectomes, connectome_graphs)):
#     edge_color_dict = {}
#     edge_width_dict = {}
#     edge_alpha_dict = {}
    
#     # Get all edges that exist in THIS connectome's graph
#     for source, target in G_conn.edges():
#         # Get connection strength from THIS connectome
#         strength = float(conn[source, target])
        
#         strength_val = float(strength * 5)
#         alpha_strength_val = min(strength * 5, 1.0)  # Cap alpha at 1.0

#         # Classify the edge
#         if source < 50 and target < 50:
#             # Within left hemisphere
#             edge_color_dict[(source, target)] = within_left_color
#             edge_width_dict[(source, target)] = strength_val
#             edge_alpha_dict[(source, target)] = alpha_strength_val
#         elif source >= 50 and target >= 50:
#             # Within right hemisphere
#             edge_color_dict[(source, target)] = within_right_color
#             edge_width_dict[(source, target)] = strength_val
#             edge_alpha_dict[(source, target)] = alpha_strength_val
#         else:
#             # Between hemispheres - emphasize these
#             edge_color_dict[(source, target)] = between_hemispheres_color
#             edge_width_dict[(source, target)] = strength_val
#             edge_alpha_dict[(source, target)] = alpha_strength_val
    
#     edge_color_dicts.append(edge_color_dict)
#     edge_width_dicts.append(edge_width_dict)
#     edge_alpha_dicts.append(edge_alpha_dict)

# # ============================================
# # Optional: Arrange nodes to separate hemispheres
# # ============================================
# node_positions_separated = {}
# separation = 2.5  # Adjust this value to increase/decrease separation

# for i in range(n_nodes):
#     x, y = node_layout[i]
#     if i < 50:
#         # Shift left hemisphere nodes to the left
#         node_positions_separated[i] = np.array([x - separation, y])
#     else:
#         # Shift right hemisphere nodes to the right
#         node_positions_separated[i] = np.array([x + separation, y])

# # ============================================
# # Analyze inter vs intra hemispheric connectivity
# # ============================================
# print("\nConnectivity Analysis:")
# for idx, (conn, label) in enumerate(zip(connectomes, labels)):
#     # Calculate statistics
#     within_left = conn[:50, :50]
#     within_right = conn[50:, 50:]
#     between = np.concatenate([conn[:50, 50:].flatten(), conn[50:, :50].flatten()])
    
#     print(f"{label}:")
#     print(f"  Within left hemisphere:  mean={within_left.mean():.4f}, std={within_left.std():.4f}")
#     print(f"  Within right hemisphere: mean={within_right.mean():.4f}, std={within_right.std():.4f}")
#     print(f"  Between hemispheres:     mean={between.mean():.4f}, std={between.std():.4f}")
#     print(f"  Ratio (between/within):  {between.mean() / ((within_left.mean() + within_right.mean())/2):.4f}")

# # ============================================
# # Create visualizations
# # ============================================
# fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 10))), dpi=150)
# axes = axes.flatten()

# g_arr = []

# # Process each connectome
# for idx, label in enumerate(labels):
    
#     print(f"\nCreating visualization for {label}...")
#     plt.sca(axes[idx])
    
#     # Use the CONNECTOME-SPECIFIC graph but SHARED distance-based layout
#     g = Graph(connectome_graphs[idx],  # Each connectome has its own edges
#               node_color=node_colors_arr[idx],
#               node_size=8, # 12,
#               node_edge_width=0.1, # 2,
#               edge_alpha=edge_alpha_dicts[idx],  # Different for each connectome
#               edge_color=edge_color_dicts[idx],  # Different for each connectome
#               edge_width=edge_width_dicts[idx],  # Different for each connectome!
#               node_layout=node_positions_separated,  # SAME layout for all (from distance matrix)
#               edge_layout='curved', # bundled',
#               edge_layout_kwargs=dict(k=2000),
#               ax=axes[idx], 
#               scale=(12.0, 10.0)
#     )
    
#     axes[idx].set_title(f'{label}')
#     g_arr.append(g)
    
# save_file_path = save_dir / 'connectomes_visualization_distance_layout_connectome_edges.pdf'
# plt.savefig(save_file_path, dpi=150, bbox_inches='tight')
# print(f"\nVisualization complete! Saved as '{save_file_path}'")
# plt.show()
# print(save_file_path)

# GOOD STUFF

In [ ]:
import numpy as np

# Load distance matrix
subject_id = 206
distance_matrices = np.load("/Users/adrian/Documents/01_projects/14_4D_lab/14_4D_lab_code/data/preprocessed/suarez_MaMI_dataset/02_distance_matrices/distance_matrix_50.npy")
d = distance_matrices[subject_id]  

# Normalize distance matrix
d = (d - np.min(d)) / (np.max(d) - np.min(d))

# ============================================
# Create distance-weighted graph for layout
# ============================================
print("Creating distance-weighted graph for layout...")

n_nodes = len(d)
G_distance = nx.Graph()
G_distance.add_nodes_from(range(n_nodes))

# Add edges with weights inversely proportional to distances
epsilon = 1e-10
for i in range(n_nodes):
    for j in range(i+1, n_nodes):
        # Convert distance to weight: closer nodes should attract more
        weight = 1.0 / (d[i, j] + epsilon)
        G_distance.add_edge(i, j, weight=weight)

# Compute layout based on distance matrix
print("Computing force-directed layout based on distance matrix...")
node_layout = nx.kamada_kawai_layout(G_distance)
print("Layout computed successfully!")

# Convert to numpy arrays for compatibility
for node in node_layout:
    node_layout[node] = np.array(node_layout[node])

# ============================================
# Create CONNECTOME graphs (for edges only)
# ============================================
connectome_graphs = []
for conn in connectomes:
    G_conn = nx.from_numpy_array(conn)
    connectome_graphs.append(G_conn)


In [ ]:
for conn in connectomes: 
    print((d*conn).sum())

In [ ]:

# ============================================
# Extract communities and create node colors
# ============================================
from netneurotools.modularity import consensus_modularity as netneurotools_modularity 

communities_arr = [] 
node_colors_arr = []
node_communities_arr = []

# Load colormap
# cmap = load_cmap("Antique")

for conn in connectomes:   
    communities, _, _ = netneurotools_modularity(conn)
    communities_arr.append(communities)
    
    # Create node_to_community dictionary
    node_to_community = {node_id: int(community_id) for node_id, community_id in enumerate(communities)}
    
    # Get number of unique communities and their sizes
    unique_communities = np.unique(communities)
    num_communities = len(unique_communities)
    partition_sizes = [np.sum(communities == comm) for comm in unique_communities]
    
    print(f"  Found {num_communities} communities")
    print(f"  Community sizes: {partition_sizes}")
    
    # Create color mapping
    # node_color = {node_id: "black" # cmap((community_id - 1) / max(num_communities - 1, 1)) 
    #               for node_id, community_id in node_to_community.items()}
    
    # node_colors_arr.append(node_color)
    # node_communities_arr.append(node_to_community)

# Define colors for the two groups of nodes
color_first_50 = "#FFFCF2" # default_colors["warms"]["YELLOW"]  # Left hemisphere
color_last_50 = "#b8b8b8" # default_colors["greens"]["LUXUARY_GREEN"]  # Right hemisphere

# Get the number of nodes from your graph G
num_nodes = len(G.nodes()) if hasattr(G, 'nodes') else 100

# Create node color dictionary
node_color_dict = {}
for i in range(num_nodes):
    if i < 50:
        node_color_dict[i] = color_first_50
    else:
        node_color_dict[i] = color_last_50

# Create an array of color dicts for each subplot
node_colors_arr = [node_color_dict.copy() for _ in range(len(labels))]

# Convert distance matrix to edge_length dictionary (for layout only)
edge_length = {}
for source, target in G.edges():
    edge_length[(source, target)] = d[source, target]/10


# ============================================
# Create edge properties for EACH connectome
# ============================================
labels = ['A', 'B', 'C', 'D', 'E', 'F']
edge_color_dicts = []
edge_width_dicts = []
edge_alpha_dicts = []

# Define colors for different connection types
within_color = "#232324" # HALF BLACK within hemisphere
between_hemispheres_color ="#3FA5C4" # '#591154' # #9B59B6'  # PURPLER for cross-hemisphere connections

# First, collect all strengths to find min and max for normalization
all_strengths = []
for conn in connectomes:
    for i in range(len(conn)):
        for j in range(i+1, len(conn)):
            if conn[i, j] > 0:  # Only non-zero connections
                all_strengths.append(conn[i, j])
                
min_strength = float(np.min(all_strengths))
max_strength = float(np.max(all_strengths))
print(min_strength, max_strength)

print(f"Strength range: {min_strength:.4f} to {max_strength:.4f}")

for conn_idx, (conn, G_conn) in enumerate(zip(connectomes, connectome_graphs)):
    edge_color_dict = {}
    edge_width_dict = {}
    edge_alpha_dict = {}
    
    # Get all edges that exist in THIS connectome's graph
    for source, target in G_conn.edges():
        # Get connection strength from THIS connectome
        # strength = float(conn[source, target])
        
        # # Normalize strength to [0, 1] range
        # normalized_strength = (strength - min_strength) / (max_strength - min_strength + 1e-10)
        
        # # Map normalized strength to alpha range [0.2, 0.8]
        # alpha_val = 0.4 + (normalized_strength * 0.6)  # Maps [0,1] to [0.4, 1.0]

        # # Map normalized strength to width range (adjust multipliers as needed)
        # width_val = float(1 + (normalized_strength * 6))  # Maps [0,1] to [1.0, 7.0]

        # Classify the edge
        factor = 0.5
        if (source < 50 and target < 50) or (source >= 50 and target >= 50):
            # Within hemisphere
            edge_color_dict[(source, target)] = within_color
            edge_width_dict[(source, target)] = 2.0 * factor
            edge_alpha_dict[(source, target)] = 1.0 * factor
        else:
            # Between hemispheres - emphasize these
            edge_color_dict[(source, target)] = between_hemispheres_color
            edge_width_dict[(source, target)] = 3.0 * factor
            edge_alpha_dict[(source, target)] = 0.5 * factor
    
    edge_color_dicts.append(edge_color_dict)
    edge_width_dicts.append(edge_width_dict)
    edge_alpha_dicts.append(edge_alpha_dict)

# ============================================
# Optional: Arrange nodes to separate hemispheres
# ============================================
node_positions_separated = {}
separation = 0  # Adjust this value to increase/decrease separation

for i in range(n_nodes):
    x, y = node_layout[i]
    if i < 50:
        # Shift left hemisphere nodes to the left
        node_positions_separated[i] = np.array([x - separation, y])
    else:
        # Shift right hemisphere nodes to the right
        node_positions_separated[i] = np.array([x + separation, y])


# ============================================
# Create visualizations
# ============================================
fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 10))), dpi=150)
axes = axes.flatten()

g_arr = []

# Process each connectome
for idx, (conn, label) in enumerate(zip(connectomes, labels)):
    
    # Calculate statistics
    within_left = conn[:50, :50]
    within_right = conn[50:, 50:]
    between = np.concatenate([conn[:50, 50:].flatten(), conn[50:, :50].flatten()])
    
    print(f"{label}:")
    print(f"  Within left hemisphere:  mean={within_left.mean():.4f}, std={within_left.std():.4f}")
    print(f"  Within right hemisphere: mean={within_right.mean():.4f}, std={within_right.std():.4f}")
    print(f"  Between hemispheres:     mean={between.mean():.4f}, std={between.std():.4f}")
    print(f"  Ratio (between/within):  {between.mean() / ((within_left.mean() + within_right.mean())/2):.4f}")



    print(f"\nCreating visualization for {label}...")
    plt.sca(axes[idx])
    

    # Use the CONNECTOME-SPECIFIC graph but SHARED distance-based layout
    g = Graph(connectome_graphs[idx],  # Each connectome has its own edges
              node_color=node_colors_arr[idx],
              node_size=4,
              node_edge_width=0.1,
              edge_alpha=edge_alpha_dicts[idx],  # Different for each connectome, scaled [0.2, 0.8]
              edge_color=edge_color_dicts[idx],  # Different for each connectome
              edge_width=edge_width_dicts[idx],  # Different for each connectome, correlated with alpha
              node_layout=node_positions_separated,  # SAME layout for all (from distance matrix)
              edge_layout='straight', # bundled',
              edge_layout_kwargs=dict(k=2000),
              ax=axes[idx], 
              scale=(12.0, 10.0)
    )
    
    # axes[idx].set_title(f'{label}\nRatio (between/within): {between.mean() / ((within_left.mean() + within_right.mean())/2):.4f}')
    axes[idx].set_title(f'{label}\nCost: {(d*conn).sum():.0f}')
    g_arr.append(g)
    
save_file_path = save_dir / 'connectomes_visualization_distance_layout_connectome_edges_light_colors_no_separation_ratio.pdf'
plt.savefig(save_file_path, dpi=150, bbox_inches='tight')
print(f"\nVisualization complete! Saved as '{save_file_path}'")
plt.show()
print(save_file_path)

In [ ]:

# ============================================
# Extract communities and create node colors
# ============================================
from netneurotools.modularity import consensus_modularity as netneurotools_modularity 

communities_arr = [] 
node_colors_arr = []
node_communities_arr = []

# Load colormap
# cmap = load_cmap("Antique")

for conn in connectomes:   
    communities, _, _ = netneurotools_modularity(conn)
    communities_arr.append(communities)
    
    # Create node_to_community dictionary
    node_to_community = {node_id: int(community_id) for node_id, community_id in enumerate(communities)}
    
    # Get number of unique communities and their sizes
    unique_communities = np.unique(communities)
    num_communities = len(unique_communities)
    partition_sizes = [np.sum(communities == comm) for comm in unique_communities]
    
    print(f"  Found {num_communities} communities")
    print(f"  Community sizes: {partition_sizes}")
    
    # Create color mapping
    # node_color = {node_id: "black" # cmap((community_id - 1) / max(num_communities - 1, 1)) 
    #               for node_id, community_id in node_to_community.items()}
    
    # node_colors_arr.append(node_color)
    # node_communities_arr.append(node_to_community)

# Define colors for the two groups of nodes
color_first_50 = "#FFFCF2" # default_colors["warms"]["YELLOW"]  # Left hemisphere
color_last_50 = "#b8b8b8" # default_colors["greens"]["LUXUARY_GREEN"]  # Right hemisphere

# Get the number of nodes from your graph G
num_nodes = len(G.nodes()) if hasattr(G, 'nodes') else 100

# Create node color dictionary
node_color_dict = {}
for i in range(num_nodes):
    if i < 50:
        node_color_dict[i] = color_first_50
    else:
        node_color_dict[i] = color_last_50

# Create an array of color dicts for each subplot
node_colors_arr = [node_color_dict.copy() for _ in range(len(labels))]

# Convert distance matrix to edge_length dictionary (for layout only)
edge_length = {}
for source, target in G.edges():
    edge_length[(source, target)] = d[source, target]/10


# ============================================
# Create edge properties for EACH connectome
# ============================================
labels = ['A', 'B', 'C', 'D', 'E', 'F']
edge_color_dicts = []
edge_width_dicts = []
edge_alpha_dicts = []

# Define colors for different connection types
within_color = "#232324" # HALF BLACK within hemisphere
between_hemispheres_color ="#3FA5C4" # '#591154' # #9B59B6'  # PURPLER for cross-hemisphere connections

# First, collect all strengths to find min and max for normalization
all_strengths = []
for conn in connectomes:
    for i in range(len(conn)):
        for j in range(i+1, len(conn)):
            if conn[i, j] > 0:  # Only non-zero connections
                all_strengths.append(conn[i, j])
                
min_strength = float(np.min(all_strengths))
max_strength = float(np.max(all_strengths))
print(min_strength, max_strength)

print(f"Strength range: {min_strength:.4f} to {max_strength:.4f}")

for conn_idx, (conn, G_conn) in enumerate(zip(connectomes, connectome_graphs)):
    edge_color_dict = {}
    edge_width_dict = {}
    edge_alpha_dict = {}
    
    # Get all edges that exist in THIS connectome's graph
    for source, target in G_conn.edges():
        # Get connection strength from THIS connectome
        # strength = float(conn[source, target])
        
        # # Normalize strength to [0, 1] range
        # normalized_strength = (strength - min_strength) / (max_strength - min_strength + 1e-10)
        
        # # Map normalized strength to alpha range [0.2, 0.8]
        # alpha_val = 0.4 + (normalized_strength * 0.6)  # Maps [0,1] to [0.4, 1.0]

        # # Map normalized strength to width range (adjust multipliers as needed)
        # width_val = float(1 + (normalized_strength * 6))  # Maps [0,1] to [1.0, 7.0]

        # Classify the edge
        factor = 0.5
        if (source < 50 and target < 50) or (source >= 50 and target >= 50):
            # Within hemisphere
            edge_color_dict[(source, target)] = within_color
            edge_width_dict[(source, target)] = 2.0 * factor
            edge_alpha_dict[(source, target)] = 1.0 * factor
        else:
            # Between hemispheres - emphasize these
            edge_color_dict[(source, target)] = between_hemispheres_color
            edge_width_dict[(source, target)] = 3.0 * factor
            edge_alpha_dict[(source, target)] = 0.5 * factor
    
    edge_color_dicts.append(edge_color_dict)
    edge_width_dicts.append(edge_width_dict)
    edge_alpha_dicts.append(edge_alpha_dict)

# ============================================
# Optional: Arrange nodes to separate hemispheres
# ============================================
node_positions_separated = {}
separation = 1  # Adjust this value to increase/decrease separation

for i in range(n_nodes):
    x, y = node_layout[i]
    if i < 50:
        # Shift left hemisphere nodes to the left
        node_positions_separated[i] = np.array([x - separation, y])
    else:
        # Shift right hemisphere nodes to the right
        node_positions_separated[i] = np.array([x + separation, y])

# ============================================
# Analyze inter vs intra hemispheric connectivity
# ============================================
print("\nConnectivity Analysis:")
for idx, (conn, label) in enumerate(zip(connectomes, labels)):
    # Calculate statistics
    within_left = conn[:50, :50]
    within_right = conn[50:, 50:]
    between = np.concatenate([conn[:50, 50:].flatten(), conn[50:, :50].flatten()])
    
    print(f"{label}:")
    print(f"  Within left hemisphere:  mean={within_left.mean():.4f}, std={within_left.std():.4f}")
    print(f"  Within right hemisphere: mean={within_right.mean():.4f}, std={within_right.std():.4f}")
    print(f"  Between hemispheres:     mean={between.mean():.4f}, std={between.std():.4f}")
    print(f"  Ratio (between/within):  {between.mean() / ((within_left.mean() + within_right.mean())/2):.4f}")

# ============================================
# Create visualizations
# ============================================
fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 10))), dpi=150)
axes = axes.flatten()

g_arr = []

# Process each connectome
for idx, label in enumerate(labels):
    
    print(f"\nCreating visualization for {label}...")
    plt.sca(axes[idx])
    
    # Use the CONNECTOME-SPECIFIC graph but SHARED distance-based layout
    g = Graph(connectome_graphs[idx],  # Each connectome has its own edges
              node_color=node_colors_arr[idx],
              node_size=4,
              node_edge_width=0.1,
              edge_alpha=edge_alpha_dicts[idx],  # Different for each connectome, scaled [0.2, 0.8]
              edge_color=edge_color_dicts[idx],  # Different for each connectome
              edge_width=edge_width_dicts[idx],  # Different for each connectome, correlated with alpha
              node_layout=node_positions_separated,  # SAME layout for all (from distance matrix)
              edge_layout='straight', # bundled',
              edge_layout_kwargs=dict(k=2000),
              ax=axes[idx], 
              scale=(12.0, 10.0)
    )
    
    axes[idx].set_title(f'{label}')
    g_arr.append(g)
    
save_file_path = save_dir / 'connectomes_visualization_distance_layout_connectome_edges_light_colors.pdf'
plt.savefig(save_file_path, dpi=150, bbox_inches='tight')
print(f"\nVisualization complete! Saved as '{save_file_path}'")
plt.show()
print(save_file_path)

In [ ]:
# # visualize node layout 
# for idx, (x, y) in enumerate(node_layout.values()):
#     plt.scatter(x, y)

In [ ]:
# # Create a NetworkX graph from your adjacency matrix
# G = nx.from_numpy_array(connectomes[0])

# # Detect communities using the Louvain method (fast and effective)
# communities_generator = community.greedy_modularity_communities(G)
# communities = list(communities_generator)

# # Step 3: Create node_to_community dictionary
# node_to_community = {}
# for community_id, community_nodes in enumerate(communities):
#     for node in community_nodes:
#         node_to_community[node] = community_id

# # Step 4: Get partition sizes
# partition_sizes = [len(comm) for comm in communities]
# print(f"Found {len(communities)} communities")
# print(f"Community sizes: {partition_sizes}")

# # Load colormap
# cmap = load_cmap("AlbrechtDurer")
# num_communities = len(communities)

# # Map community IDs to colors from the colormap
# node_color = {node: cmap(community_id / max(num_communities - 1, 1)) 
#               for node, community_id in node_to_community.items()}


# print("Creating visualization with distance-weighted force-directed layout...")

# # Visualize the original connectome graph with distance-based positions
# g = Graph(G,
#       node_color=node_color, 
#       node_edge_width=0.1,
#       edge_alpha=0.1,
#       node_layout=node_layout,  # Use distance-weighted positions
#       edge_layout='bundled', 
#       edge_layout_kwargs=dict(k=2000),
# )

# plt.title("Connectome with Distance-Weighted Force-Directed Layout")
# plt.show()

In [ ]:
from netneurotools.modularity import consensus_modularity as netneurotools_modularity 

communities_arr = [] 
node_colors_arr = []
node_communities_arr = []

for conn in connectomes:   
    communities, _, _ = netneurotools_modularity(conn)
    communities_arr.append(communities)

    # Create a NetworkX graph from the adjacency matrix
    G = nx.from_numpy_array(conn)
    
    # Get the community assignments for this connectome
    # communities = communities_arr[idx]
    
    # Create node_to_community dictionary
    # communities is already an array where index = node_id, value = community_id
    node_to_community = {node_id: int(community_id) for node_id, community_id in enumerate(communities)}
    
    # Get number of unique communities and their sizes
    unique_communities = np.unique(communities)
    num_communities = len(unique_communities)
    partition_sizes = [np.sum(communities == comm) for comm in unique_communities]
    
    print(f"  Found {num_communities} communities")
    print(f"  Community sizes: {partition_sizes}")
    
    # Create color mapping
    # Normalize community IDs to [0, 1] range for colormap
    node_color = {node_id: cmap((community_id - 1) / max(num_communities - 1, 1)) 
                  for node_id, community_id in node_to_community.items()}
    
    node_colors_arr.append(node_color)
    node_communities_arr.append(node_to_community)


In [ ]:
# import numpy as np
# labels = ['A', 'B', 'C', 'D', 'E', 'F']
# g_arr = []

# # Define colors for the two groups of nodes
# color_first_50 = default_colors["warms"]["YELLOW"]  # Left hemisphere
# color_last_50 = default_colors["greens"]["LUXUARY_GREEN"]  # Right hemisphere

# # Get the number of nodes from your graph G
# num_nodes = len(G.nodes()) if hasattr(G, 'nodes') else 100

# # Create node color dictionary
# node_color_dict = {}
# for i in range(num_nodes):
#     if i < 50:
#         node_color_dict[i] = color_first_50
#     else:
#         node_color_dict[i] = color_last_50

# # Create an array of color dicts for each subplot
# node_colors_arr = [node_color_dict.copy() for _ in range(len(labels))]

# # Convert distance matrix to edge_length dictionary (for layout only)
# # edge_length = {}
# # for source, target in G.edges():
# #     edge_length[(source, target)] = d[source, target]/10

# # ============================================
# # Create edge properties for EACH connectome
# # ============================================
# edge_color_dicts = []
# edge_width_dicts = []
# edge_alpha_dicts = []

# # Define colors for different connection types
# within_left_color = "black"  # Black edges within left hemisphere
# within_right_color = "black" # color_last_50  # Green edges within right hemisphere
# between_hemispheres_color = '#9B59B6'  # Purple for cross-hemisphere connections

# for conn_idx, conn in enumerate(connectomes):
#     edge_color_dict = {}
#     edge_width_dict = {}
#     edge_alpha_dict = {}
    
#     # Get all edges that exist in the graph structure
#     for source, target in G.edges():
#         # Get connection strength from THIS connectome - CONVERT TO PYTHON FLOAT
#         strength = float(conn[source, target])
        
#         strength_val = float(strength * 5)
#         alpha_strength_val = min(strength * 5, 1.0)  # Cap alpha at 1.0

#         # Classify the edge
#         if source < 50 and target < 50:
#             # Within left hemisphere
#             edge_color_dict[(source, target)] = within_left_color
#             edge_width_dict[(source, target)] = strength_val  # Scale as needed
#             edge_alpha_dict[(source, target)] = alpha_strength_val
#         elif source >= 50 and target >= 50:
#             # Within right hemisphere
#             edge_color_dict[(source, target)] = within_right_color
#             edge_width_dict[(source, target)] = strength_val
#             edge_alpha_dict[(source, target)] = alpha_strength_val
#         else:
#             # Between hemispheres - emphasize these
#             edge_color_dict[(source, target)] = between_hemispheres_color
#             edge_width_dict[(source, target)] = strength_val
#             edge_alpha_dict[(source, target)] = alpha_strength_val
    
#     edge_color_dicts.append(edge_color_dict)
#     edge_width_dicts.append(edge_width_dict)
#     edge_alpha_dicts.append(edge_alpha_dict)

# # ============================================
# # COMPUTE GEOMETRIC LAYOUT ONCE
# # ============================================
# # print("Computing geometric layout (this may take a moment)...")
# # from netgraph import get_geometric_layout

# # # Calculate the layout once with your edge lengths
# # node_positions = get_geometric_layout(
# #     list(G.edges()),
# #     nodes=list(G.nodes()),
# #     edge_length=edge_length, 
# #     tol=1e-3, 
# #     scale=(12.0, 9.0), 
# #     origin=(0, 0), 
# # )


# print("Computing ForceAtlas2 layout (this may take a moment)...")

# # Use ForceAtlas2 layout from NetworkX
# node_positions = nx.forceatlas2_layout(
#     G,
#     pos=None,  # Start from random positions
#     max_iter=100, # =2000,  # Number of iterations (more = better convergence)
#     # linlog=False,  # Use linear model (False) or log model (True)
#     # nohubs=False,  # Prevent hubs from being pushed to the borders
#     # kr=0.001,  # Repulsion parameter
#     # dim=2,  # 2D layout
#     # seed=42  # For reproducibility
# )

# # Convert positions to numpy arrays for compatibility
# for node in node_positions:
#     node_positions[node] = np.array(node_positions[node])


# # ============================================
# # Arrange nodes spatially to separate hemispheres
# # ============================================
# node_positions_separated = {}
# separation = 3.0  # Adjust this value to increase/decrease separation

# for i in range(num_nodes):
#     x, y = node_positions[i]
#     if i < 50:
#         # Shift left hemisphere nodes to the left
#         node_positions_separated[i] = np.array([x - separation, y])
#     else:
#         # Shift right hemisphere nodes to the right
#         node_positions_separated[i] = np.array([x + separation, y])

# # ============================================
# # Analyze inter vs intra hemispheric connectivity
# # ============================================
# print("\nConnectivity Analysis:")
# for idx, (conn, label) in enumerate(zip(connectomes, labels)):
#     # Calculate statistics
#     within_left = conn[:50, :50]
#     within_right = conn[50:, 50:]
#     between = np.concatenate([conn[:50, 50:].flatten(), conn[50:, :50].flatten()])
    
#     print(f"{label}:")
#     print(f"  Within left hemisphere:  mean={within_left.mean():.4f}, std={within_left.std():.4f}")
#     print(f"  Within right hemisphere: mean={within_right.mean():.4f}, std={within_right.std():.4f}")
#     print(f"  Between hemispheres:     mean={between.mean():.4f}, std={between.std():.4f}")
#     print(f"  Ratio (between/within):  {between.mean() / ((within_left.mean() + within_right.mean())/2):.4f}")

# # Create figure with 6 subplots (2 rows x 3 columns)
# fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 10))), dpi=150)
# axes = axes.flatten()

# # Load colormap once
# cmap = load_cmap("Antique")

# # Process each connectome
# for idx, label in enumerate(labels):
    
#     print(f"\nCreating visualization for {label}...")
#     plt.sca(axes[idx])
    
#     # Use the edge properties for THIS specific connectome
#     g = Graph(G,
#               node_color=node_colors_arr[idx],
#               node_size=12,
#               node_edge_width=0.3,
#               edge_alpha=edge_alpha_dicts[idx],  # Different for each connectome
#               edge_color=edge_color_dicts[idx],  # Different for each connectome
#               edge_width=edge_width_dicts[idx],  # Different for each connectome!
#               node_layout=node_positions_separated,
#               edge_layout='curved', # bundled',
#               edge_layout_kwargs=dict(k=2000),
#               ax=axes[idx], 
#               scale=(12.0, 10.0)
#     )
    
#     axes[idx].set_title(f'{label}')
#     g_arr.append(g)
    
# save_file_path = save_dir / 'connectomes_visualization_hemispheres_varying_strength.pdf'
# plt.savefig(save_file_path, dpi=150, bbox_inches='tight')
# print(f"\nVisualization complete! Saved as '{save_file_path}'")
# plt.show()
# print(save_file_path)

In [ ]:
# import numpy as np
# labels = ['A', 'B', 'C', 'D', 'E', 'F']
# g_arr = []

# # Define colors for the two groups of nodes
# color_first_50 = default_colors["warms"]["YELLOW"]
# color_last_50 = default_colors["greens"]["LUXUARY_GREEN"]

# # Get the number of nodes from your graph G
# num_nodes = len(G.nodes()) if hasattr(G, 'nodes') else 100

# # Create node color dictionary
# node_color_dict = {}
# for i in range(num_nodes):
#     if i < 50:
#         node_color_dict[i] = color_first_50
#     else:
#         node_color_dict[i] = color_last_50

# # Create an array of color dicts for each subplot
# node_colors_arr = [node_color_dict.copy() for _ in range(len(labels))]

# # Convert distance matrix to edge_length dictionary
# edge_length = {}
# for source, target in G.edges():
#     edge_length[(source, target)] = d[source, target]/10

# # ============================================
# # COMPUTE GEOMETRIC LAYOUT ONCE
# # ============================================
# print("Computing geometric layout (this may take a moment)...")
# from netgraph import get_geometric_layout

# # Calculate the layout once with your edge lengths
# # Pass the edges as a list, not the graph object
# node_positions = get_geometric_layout(
#     list(G.edges()),  # Convert edges to list
#     nodes=list(G.nodes()),  # Also pass the nodes
#     edge_length=edge_length, 
#     tol=1e-3, 
#     scale=(10.0, 10.0), 
#     origin=(0, 0), 
# )
# print("Layout computed successfully!")

# # Create figure with 6 subplots (2 rows x 3 columns)
# fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 9))), dpi=150)
# axes = axes.flatten()

# # Load colormap once
# cmap = load_cmap("Antique")

# # Process each connectome
# for idx, label in enumerate(labels):
    
#     print(f"  Creating visualization for {label}...")
#     plt.sca(axes[idx])
    
#     # Use pre-computed node_positions instead of recalculating
#     g = Graph(G,
#               node_color=node_colors_arr[idx],
#               node_size=10,
#               node_edge_width=0.1, 
#               edge_alpha=0.3,
#               edge_width=2,
#               node_layout=node_positions,  # Pass pre-computed positions directly
#               edge_layout='bundled',
#               edge_layout_kwargs=dict(k=2000),
#               ax=axes[idx], 
#               scale=(10.0, 10.0)
#     )
    
#     axes[idx].set_title(f'{label}')
#     g_arr.append(g)
    
# save_file_path = save_dir / 'connectomes_visualization_smaller_geometric.pdf'
# plt.savefig(save_file_path, dpi=150, bbox_inches='tight')
# print(f"\nVisualization complete! Saved as '{save_file_path}'")
# plt.show()

In [ ]:
# import numpy as np

# labels = ['A', 'B', 'C', 'D', 'E', 'F']
# g_arr = []

# # Define colors for the two groups of nodes
# # color_first_50 = '#006685'   # DEEP_BLUE - change this hex code as needed
# # color_last_50 = '#E84653'    # LECKER_RED - change this hex code as needed
# color_first_50 = default_colors["warms"]["YELLOW"] # '#FF6B6B'  # Red/coral color - change this hex code as needed
# color_last_50 = default_colors["greens"]["LUXUARY_GREEN"]   # Teal color - change this hex code as needed

# # Assuming you have a distance_matrix of shape (100, 100)
# # distance_matrix = your_distance_matrix_here

# # Get the number of nodes from your graph G
# num_nodes = len(G.nodes()) if hasattr(G, 'nodes') else 100

# # Create node color dictionary
# node_color_dict = {}
# for i in range(num_nodes):
#     if i < 50:
#         node_color_dict[i] = color_first_50
#     else:
#         node_color_dict[i] = color_last_50

# # Create an array of color dicts for each subplot
# node_colors_arr = [node_color_dict.copy() for _ in range(len(labels))]

# # Convert distance matrix to edge_length dictionary
# # Only include edges that exist in your graph G
# edge_length = {}
# for source, target in G.edges():
#     # Assuming nodes are numbered 0-99 and distance_matrix is indexed the same way
#     edge_length[(source, target)] = d[source, target]/10

# # Create figure with 6 subplots (2 rows x 3 columns)
# fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 9))), dpi=150)
# axes = axes.flatten()

# # Load colormap once
# cmap = load_cmap("Antique")

# # Process each connectome
# for idx, label in enumerate(labels):
    
#     print(f"  Creating visualization for {label}...")
#     plt.sca(axes[idx])
    
#     g = Graph(G,
#               node_color=node_colors_arr[idx],
#               node_size=8,  # Increased size - adjust as needed (try 10-15 for larger)
#               node_edge_width=0.1, 
#               edge_alpha=0.1,
#               edge_width=2,
#               node_layout='geometric', 
#               node_layout_kwargs=dict(edge_length=edge_length, tol=1e-3),  # Pass edge lengths
#               edge_layout='bundled',
#               edge_layout_kwargs=dict(k=2000),
#               ax=axes[idx], 
#               scale=(10.0, 10.0)
#     )
    
#     axes[idx].set_title(f'{label}')
#     g_arr.append(g)
    
# save_file_path = save_dir / 'connectomes_visualization_smaller_geometric.pdf'
# plt.savefig(save_file_path, dpi=150, bbox_inches='tight')
# print(f"\nVisualization complete! Saved as '{save_file_path}'")
# plt.show()

In [ ]:
# labels = ['A', 'B', 'C', 'D', 'E', 'F']
# g_arr = []

# # Define colors for the two groups of nodes
# # color_first_50 = '#006685'   # DEEP_BLUE - change this hex code as needed
# # color_last_50 = '#E84653'    # LECKER_RED - change this hex code as needed
# color_first_50 = default_colors["warms"]["YELLOW"] # '#FF6B6B'  # Red/coral color - change this hex code as needed
# color_last_50 = default_colors["greens"]["LUXUARY_GREEN"]   # Teal color - change this hex code as needed

# # Assuming G has 100 nodes numbered 0-99
# # Create a dictionary mapping node IDs to colors
# num_nodes = len(G.nodes()) if hasattr(G, 'nodes') else 100  # Adjust based on your graph
# node_color_dict = {}
# for i in range(num_nodes):
#     if i < 50:
#         node_color_dict[i] = color_first_50
#     else:
#         node_color_dict[i] = color_last_50

# # If you need different colors for each subplot, create an array of dicts
# node_colors_arr = [node_color_dict.copy() for _ in range(len(labels))]

# # Create figure with 6 subplots (2 rows x 3 columns)
# fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 9))), dpi=150)
# axes = axes.flatten()  # Flatten to make indexing easier

# # Load colormap once
# cmap = load_cmap("Antique")

# # Process each connectome
# for idx, label in enumerate(labels):
    
#     # Create visualization in the corresponding subplot
#     print(f"  Creating visualization...")
#     plt.sca(axes[idx])  # Set current axis
    
#     g = Graph(G,
#               node_color=node_colors_arr[idx],  # Now a dict mapping nodes to colors
#               node_size=5, # 8,  # Increased from default - adjust as needed (try 10-15 for larger)
#               node_edge_width=0.1, 
#               edge_alpha=0.1,
#               edge_width=2,
#               node_layout=node_layout,
#               edge_layout='bundled',
#               edge_layout_kwargs=dict(k=2000),
#               ax=axes[idx]  
#     )
    
#     axes[idx].set_title(f'{label}')
#     g_arr.append(g)
    
# save_file_path = save_dir / 'connectomes_visualization_smaller_hemiid.pdf'
# plt.savefig(save_file_path, dpi=150, bbox_inches='tight')
# print(f"\nVisualization complete! Saved as '{save_file_path}'")
# plt.show()

In [ ]:
save_file_path

# Plot the graphs

In [ ]:
# import numpy as np
# labels = ['A', 'B', 'C', 'D', 'E', 'F']
# g_arr = []

# # Define colors for the two groups of nodes
# color_first_50 = default_colors["warms"]["YELLOW"]  # Left hemisphere
# color_last_50 = default_colors["greens"]["LUXUARY_GREEN"]  # Right hemisphere

# # Get the number of nodes from your graph G
# num_nodes = len(G.nodes()) if hasattr(G, 'nodes') else 100

# # Create node color dictionary
# node_color_dict = {}
# for i in range(num_nodes):
#     if i < 50:
#         node_color_dict[i] = color_first_50
#     else:
#         node_color_dict[i] = color_last_50

# # Create an array of color dicts for each subplot
# node_colors_arr = [node_color_dict.copy() for _ in range(len(labels))]

# # Convert distance matrix to edge_length dictionary
# edge_length = {}
# for source, target in G.edges():
#     edge_length[(source, target)] = d[source, target]/10

# # ============================================
# # Edge color, width, and alpha based on hemisphere connectivity
# # ============================================
# edge_color_dict = {}
# edge_width_dict = {}
# edge_alpha_dict = {}

# # Define colors for different connection types
# within_left_color = "black" # color_first_50  # Yellow edges within left hemisphere
# within_right_color = "black" # color_last_50  # Green edges within right hemisphere
# between_hemispheres_color = '#9B59B6'  # Purple for cross-hemisphere connections

# for source, target in G.edges():
#     # Get connection strength (adjust this based on your data)
#     strength = G[source][target].get('weight', 1.0)
#     # Or use inverse distance: strength = 1.0 / d[source, target] if d[source, target] > 0 else 0
    
#     # Classify the edge
#     if source < 50 and target < 50:
#         # Within left hemisphere
#         edge_color_dict[(source, target)] = within_left_color
#         edge_width_dict[(source, target)] = strength * 2
#         edge_alpha_dict[(source, target)] = 0.2
#     elif source >= 50 and target >= 50:
#         # Within right hemisphere
#         edge_color_dict[(source, target)] = within_right_color
#         edge_width_dict[(source, target)] = strength * 2
#         edge_alpha_dict[(source, target)] = 0.2
#     else:
#         # Between hemispheres - emphasize these more
#         edge_color_dict[(source, target)] = between_hemispheres_color
#         edge_width_dict[(source, target)] = strength * 3  # Make inter-hemispheric edges thicker
#         edge_alpha_dict[(source, target)] = 0.8

# # ============================================
# # COMPUTE GEOMETRIC LAYOUT ONCE
# # ============================================
# print("Computing geometric layout (this may take a moment)...")
# from netgraph import get_geometric_layout

# # Calculate the layout once with your edge lengths
# node_positions = get_geometric_layout(
#     list(G.edges()),
#     nodes=list(G.nodes()),
#     edge_length=edge_length, 
#     tol=1e-3, 
#     scale=(10.0, 10.0), 
#     origin=(0, 0), 
# )
# print("Layout computed successfully!")

# # ============================================
# # Arrange nodes spatially to separate hemispheres
# # ============================================
# # Shift left hemisphere nodes to the left, right hemisphere to the right
# node_positions_separated = {}
# left_nodes_x = [node_positions[i][0] for i in range(50)]
# right_nodes_x = [node_positions[i][0] for i in range(50, 100)]

# # Calculate shift to create clear separation
# separation = 3.0  # Adjust this value to increase/decrease separation

# for i in range(num_nodes):
#     x, y = node_positions[i]
#     if i < 50:
#         # Shift left hemisphere nodes to the left - CONVERT TO NUMPY ARRAY
#         node_positions_separated[i] = np.array([x - separation, y])
#     else:
#         # Shift right hemisphere nodes to the right - CONVERT TO NUMPY ARRAY
#         node_positions_separated[i] = np.array([x + separation, y])

# # Create figure with 6 subplots (2 rows x 3 columns)
# fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((21, 10))), dpi=150)
# axes = axes.flatten()

# # Load colormap once
# cmap = load_cmap("Antique")

# # Process each connectome
# for idx, label in enumerate(labels):
    
#     print(f"  Creating visualization for {label}...")
#     plt.sca(axes[idx])
    
#     # Use pre-computed node_positions instead of recalculating
#     g = Graph(G,
#               node_color=node_colors_arr[idx],
#               node_size=12,  # Slightly larger nodes
#               node_edge_width=0.3,  # More visible node borders
#               edge_alpha=edge_alpha_dict,  # Use dictionary for different alphas
#               edge_color=edge_color_dict,  # Use dictionary for different colors
#               edge_width=edge_width_dict,  # Width based on connection strength
#               node_layout=node_positions_separated,  # Use separated layout
#               edge_layout='bundled',
#               edge_layout_kwargs=dict(k=2000),
#               ax=axes[idx], 
#               scale=(12.0, 10.0)  # Wider to accommodate separation
#     )
    
#     axes[idx].set_title(f'{label}')
#     g_arr.append(g)
    
# save_file_path = save_dir / 'connectomes_visualization_hemispheres.pdf'
# plt.savefig(save_file_path, dpi=150, bbox_inches='tight')
# print(f"\nVisualization complete! Saved as '{save_file_path}'")
# plt.show()

In [ ]:
# from pypalettes import load_cmap
# cmap = load_cmap("Aurora")

from pypalettes import load_cmap
cmap = load_cmap("Antique")

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from netgraph import Graph
from networkx.algorithms import community


In [ ]:
# import numpy as np
# labels = ['A', 'B', 'C', 'D', 'E', 'F']
# g_arr = []

# # Define colors for the two groups of nodes
# color_first_50 = default_colors["warms"]["YELLOW"]
# color_last_50 = default_colors["greens"]["LUXUARY_GREEN"]

# # Get the number of nodes from your graph G
# num_nodes = len(G.nodes()) if hasattr(G, 'nodes') else 100

# # Create node color dictionary
# node_color_dict = {}
# for i in range(num_nodes):
#     if i < 50:
#         node_color_dict[i] = color_first_50
#     else:
#         node_color_dict[i] = color_last_50

# # Create an array of color dicts for each subplot
# node_colors_arr = [node_color_dict.copy() for _ in range(len(labels))]

# # Convert distance matrix to edge_length dictionary
# edge_length = {}
# for source, target in G.edges():
#     edge_length[(source, target)] = d[source, target]/10

# # Create edge_width dictionary based on connection strength
# # You can adjust the multiplier (e.g., 5) to scale the widths appropriately
# edge_width_dict = {}
# for source, target in G.edges():
#     # Get the connection strength from your adjacency matrix or edge weights
#     # Assuming G has edge weights stored as 'weight' attribute
#     # Or you can use your distance matrix: strength = 1 / d[source, target]
#     if G.has_edge(source, target):
#         # Option 1: If your graph has edge weights
#         strength = G[source][target].get('weight', 1.0)
        
#         # Option 2: If using inverse distance as strength
#         # strength = 1.0 / d[source, target] if d[source, target] > 0 else 0
        
#         # Scale the width (adjust multiplier as needed)
#         edge_width_dict[(source, target)] = strength * 2  # Adjust multiplier as needed

# # ============================================
# # COMPUTE GEOMETRIC LAYOUT ONCE
# # ============================================
# print("Computing geometric layout (this may take a moment)...")
# from netgraph import get_geometric_layout

# # Calculate the layout once with your edge lengths
# # Pass the edges as a list, not the graph object
# node_positions = get_geometric_layout(
#     list(G.edges()),  # Convert edges to list
#     nodes=list(G.nodes()),  # Also pass the nodes
#     edge_length=edge_length, 
#     tol=1e-3, 
#     scale=(10.0, 10.0), 
#     origin=(0, 0), 
# )
# print("Layout computed successfully!")

# # Create figure with 6 subplots (2 rows x 3 columns)
# fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 9))), dpi=150)
# axes = axes.flatten()

# # Load colormap once
# cmap = load_cmap("Antique")

# # Process each connectome
# for idx, label in enumerate(labels):
    
#     print(f"  Creating visualization for {label}...")
#     plt.sca(axes[idx])
    
#     # Use pre-computed node_positions instead of recalculating
#     g = Graph(G,
#               node_color=node_colors_arr[idx],
#               node_size=10,
#               node_edge_width=0.1, 
#               edge_alpha=0.3,
#               edge_width=edge_width_dict,  # Pass the dictionary of edge widths
#               node_layout=node_positions,  # Pass pre-computed positions directly
#               edge_layout='bundled',
#               edge_layout_kwargs=dict(k=2000),
#               ax=axes[idx], 
#               scale=(10.0, 10.0)
#     )
    
#     axes[idx].set_title(f'{label}')
#     g_arr.append(g)
    
# save_file_path = save_dir / 'connectomes_visualization_smaller_geometric.pdf'
# plt.savefig(save_file_path, dpi=150, bbox_inches='tight')
# print(f"\nVisualization complete! Saved as '{save_file_path}'")
# plt.show()
# print(save_file_path)

In [ ]:
# import numpy as np
# labels = ['A', 'B', 'C', 'D', 'E', 'F']
# g_arr = []

# # Define colors for the two groups of nodes
# color_first_50 = default_colors["warms"]["YELLOW"]
# color_last_50 = default_colors["greens"]["LUXUARY_GREEN"]

# # Get the number of nodes from your graph G
# num_nodes = len(G.nodes()) if hasattr(G, 'nodes') else 100

# # Create node color dictionary
# node_color_dict = {}
# for i in range(num_nodes):
#     if i < 50:
#         node_color_dict[i] = color_first_50
#     else:
#         node_color_dict[i] = color_last_50

# # Create an array of color dicts for each subplot
# node_colors_arr = [node_color_dict.copy() for _ in range(len(labels))]

# # Convert distance matrix to edge_length dictionary
# edge_length = {}
# for source, target in G.edges():
#     edge_length[(source, target)] = d[source, target]/10

# # Create edge_width dictionary based on connection strength
# # You can adjust the multiplier (e.g., 5) to scale the widths appropriately
# edge_width_dict = {}
# for source, target in G.edges():
#     # Get the connection strength from your adjacency matrix or edge weights
#     # Assuming G has edge weights stored as 'weight' attribute
#     # Or you can use your distance matrix: strength = 1 / d[source, target]
#     if G.has_edge(source, target):
#         # Option 1: If your graph has edge weights
#         strength = G[source][target].get('weight', 1.0)
        
#         # Option 2: If using inverse distance as strength
#         # strength = 1.0 / d[source, target] if d[source, target] > 0 else 0
        
#         # Scale the width (adjust multiplier as needed)
#         edge_width_dict[(source, target)] = strength * 2  # Adjust multiplier as needed

# # ============================================
# # COMPUTE GEOMETRIC LAYOUT ONCE
# # ============================================
# print("Computing geometric layout (this may take a moment)...")
# from netgraph import get_geometric_layout

# # Calculate the layout once with your edge lengths
# # Pass the edges as a list, not the graph object
# node_positions = get_geometric_layout(
#     list(G.edges()),  # Convert edges to list
#     nodes=list(G.nodes()),  # Also pass the nodes
#     edge_length=edge_length, 
#     tol=1e-3, 
#     scale=(10.0, 10.0), 
#     origin=(0, 0), 
# )
# print("Layout computed successfully!")

# # Create figure with 6 subplots (2 rows x 3 columns)
# fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 9))), dpi=150)
# axes = axes.flatten()

# # Load colormap once
# cmap = load_cmap("Antique")

# # Process each connectome
# for idx, label in enumerate(labels):
    
#     print(f"  Creating visualization for {label}...")
#     plt.sca(axes[idx])
    
#     # Use pre-computed node_positions instead of recalculating
#     g = Graph(G,
#               node_color=node_colors_arr[idx],
#               node_size=10,
#               node_edge_width=0.1, 
#               edge_alpha=0.3,
#               edge_width=edge_width_dict,  # Pass the dictionary of edge widths
#               node_layout=node_positions,  # Pass pre-computed positions directly
#               edge_layout='bundled',
#               edge_layout_kwargs=dict(k=2000),
#               ax=axes[idx], 
#               scale=(10.0, 10.0)
#     )
    
#     axes[idx].set_title(f'{label}')
#     g_arr.append(g)
    
# save_file_path = save_dir / 'connectomes_visualization_smaller_geometric.pdf'
# plt.savefig(save_file_path, dpi=150, bbox_inches='tight')
# print(f"\nVisualization complete! Saved as '{save_file_path}'")
# plt.show()

In [ ]:
# labels = ['A', 'B', 'C', 'D', 'E', 'F']
# g_arr = []

# # Create figure with 6 subplots (2 rows x 3 columns)
# fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 7))), dpi=150)
# axes = axes.flatten()  # Flatten to make indexing easier

# # Load colormap once
# # cmap = load_cmap("AlbrechtDurer")
# # from pypalettes import load_cmap
# cmap = load_cmap("Antique")

# # Process each connectome
# for idx, label in enumerate(labels):
#     print(f"\nProcessing connectome {idx + 1}/6...")
    
#     # Create a NetworkX graph from the adjacency matrix
#     G = nx.from_numpy_array(connectomes[idx])
    
#     # Get the community assignments for this connectome
#     communities = communities_arr[idx]
    
#     # Create node_to_community dictionary
#     # communities is already an array where index = node_id, value = community_id
#     node_to_community = {node_id: int(community_id) for node_id, community_id in enumerate(communities)}
    
#     # Get number of unique communities and their sizes
#     unique_communities = np.unique(communities)
#     num_communities = len(unique_communities)
#     partition_sizes = [np.sum(communities == comm) for comm in unique_communities]
    
#     print(f"  Found {num_communities} communities")
#     print(f"  Community sizes: {partition_sizes}")
    
#     # Create color mapping
#     # Normalize community IDs to [0, 1] range for colormap
#     node_color = {node_id: cmap((community_id - 1) / max(num_communities - 1, 1)) 
#                   for node_id, community_id in node_to_community.items()}
    
#     # Create visualization in the corresponding subplot
#     print(f"  Creating visualization...")
#     plt.sca(axes[idx])  # Set current axis
    
#     g = Graph(G,
#               node_color=node_color, 
#               node_edge_width=0.1, 
#               edge_alpha=0.1,
#               node_layout='community', 
#               node_layout_kwargs=dict(node_to_community=node_to_community),
#               edge_layout='bundled', 
#               edge_layout_kwargs=dict(k=2000),
#               ax=axes[idx]  # Specify which subplot to use
#     )
    
#     axes[idx].set_title(f'{label}') # \n({num_communities} communities)')
#     g_arr.append(g)
    

# plt.savefig('connectomes_visualization_smaller.pdf', dpi=150, bbox_inches='tight')
# print("\nVisualization complete! Saved as 'connectomes_visualization_smaller.pdf'")
# plt.show() # 3 min 48

In [ ]:
# labels = ['A', 'B', 'C', 'D', 'E', 'F']
# g_arr = []

# # Create figure with 6 subplots (2 rows x 3 columns)
# fig, axes = plt.subplots(2, 3, figsize=(viz.cm_to_inch((18, 9))), dpi=150)
# axes = axes.flatten()  # Flatten to make indexing easier

# # Load colormap once
# # cmap = load_cmap("AlbrechtDurer")
# # from pypalettes import load_cmap
# cmap = load_cmap("Antique")

# # Process each connectome
# for idx, label in enumerate(labels):
#     print(f"\nProcessing connectome {idx + 1}/6...")
    
#     # Create a NetworkX graph from the adjacency matrix
#     G = nx.from_numpy_array(connectomes[idx])
    
#     # Get the community assignments for this connectome
#     communities = communities_arr[idx]
    
#     # Create node_to_community dictionary
#     # communities is already an array where index = node_id, value = community_id
#     node_to_community = {node_id: int(community_id) for node_id, community_id in enumerate(communities)}
    
#     # Get number of unique communities and their sizes
#     unique_communities = np.unique(communities)
#     num_communities = len(unique_communities)
#     partition_sizes = [np.sum(communities == comm) for comm in unique_communities]
    
#     print(f"  Found {num_communities} communities")
#     print(f"  Community sizes: {partition_sizes}")
    
#     # Create color mapping
#     # Normalize community IDs to [0, 1] range for colormap
#     node_color = {node_id: cmap((community_id - 1) / max(num_communities - 1, 1)) 
#                   for node_id, community_id in node_to_community.items()}
    
#     # Create visualization in the corresponding subplot
#     print(f"  Creating visualization...")
#     plt.sca(axes[idx])  # Set current axis
    
#     g = Graph(G,
#               node_color=node_color, 
#               node_edge_width=0.1, 
#               edge_alpha=0.1,
#               node_layout='community', 
#               node_layout_kwargs=dict(node_to_community=node_to_community),
#               edge_layout='bundled', 
#               edge_layout_kwargs=dict(k=2000),
#               ax=axes[idx]  # Specify which subplot to use
#     )
    
#     axes[idx].set_title(f'{label}') # \n({num_communities} communities)')
#     g_arr.append(g)
    
# file_save_path = save_dir / 'connectomes_visualization_smaller_colorful.pdf'
# plt.savefig(file_save_path, dpi=150, bbox_inches='tight')
# print(f"\nVisualization complete! Saved as '{file_save_path.name}'")
# plt.show() # 3 min 48